# Pipeline de Detecção de Crises Epilépticas — Notebook 1
## Pré-processamento e Extração de Features (FS-A)

**Dataset:** SeizeIT2 · **Autor:** Danilo Pedro da Silva Valério

---

## 0. Instalação

In [1]:
!pip install boto3 mne scipy scikit-learn PyWavelets tqdm numpy pandas matplotlib xgboost --quiet
print("✅ Dependências OK")

✅ Dependências OK



[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports

In [1]:
import os, json, warnings, gc
import numpy as np
import pandas as pd
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import mne
from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt, savgol_filter
from scipy.stats import kurtosis as scipy_kurtosis, skew as scipy_skew
import pywt
from tqdm.auto import tqdm
mne.set_log_level('WARNING')
warnings.filterwarnings('ignore')
print("✅ Imports OK")

✅ Imports OK


c:\Users\danil\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configurações Globais

In [3]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Dataset
S3_BUCKET  = 'openneuro.org'
DATASET_ID = 'ds005873'
SESSION    = 'ses-01'
SUBJECTS   = [f'sub-{i:03d}' for i in range(1, 13)]

# Sinal
SFREQ       = 250
F_HIGH_PASS = 0.5
F_NOTCH     = 50.0
F_LOW_PASS  = 40.0

# Janelas: 4 s, 5 s, 6 s com 50% de overlap
WINDOW_CONFIGS = {
    '4s_50': {'win_sec': 4, 'overlap': 0.50},
    '5s_50': {'win_sec': 5, 'overlap': 0.50},
    '6s_50': {'win_sec': 6, 'overlap': 0.50},
}
SEIZURE_OVERLAP_THRESHOLD = 0.50

# Undersampling NÃO é aplicado no Nível 2.
# Todas as janelas são salvas.
# Ratios 1:3 e 1:5 serão aplicados no Notebook 2 (apenas no treino).

# Caminhos
DATA_DIR         = './data'
RAW_DIR          = os.path.join(DATA_DIR, 'raw', DATASET_ID)
PREPROCESSED_DIR = os.path.join(DATA_DIR, 'preprocessed')
WINDOWS_DIR      = os.path.join(DATA_DIR, 'windows')
FEATURES_DIR     = os.path.join(DATA_DIR, 'features')

for d in [RAW_DIR, PREPROCESSED_DIR, WINDOWS_DIR, FEATURES_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Configurações definidas.")
print(f'   Subjects  : {SUBJECTS}')
print(f'   Janelas   : {list(WINDOW_CONFIGS.keys())}')

✅ Configurações definidas.
   Subjects  : ['sub-001', 'sub-002', 'sub-003', 'sub-004', 'sub-005', 'sub-006', 'sub-007', 'sub-008', 'sub-009', 'sub-010', 'sub-011', 'sub-012']
   Janelas   : ['4s_50', '5s_50', '6s_50']


## 3. Download dos Dados (SeizeIT2)

Download seletivo: só baixa EDFs de runs que contêm crises (`eventType` = `sz*`).

In [4]:
def get_s3_client():
    return boto3.client('s3', region_name='us-east-1',
                        config=Config(signature_version=UNSIGNED))

def list_eeg_objects(s3, subject):
    prefix = f'{DATASET_ID}/{subject}/{SESSION}/eeg/'
    paginator = s3.get_paginator('list_objects_v2')
    keys = []
    for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=prefix):
        keys.extend(o['Key'] for o in page.get('Contents', []))
    return keys

def s3_to_local(key):
    return os.path.join(DATA_DIR, 'raw', key.replace('/', os.sep))

def download_file(s3, key, local):
    if not os.path.exists(local):
        os.makedirs(os.path.dirname(local), exist_ok=True)
        s3.download_file(S3_BUCKET, key, local)

def tsv_has_seizure(tsv_path):
    try:
        df = pd.read_csv(tsv_path, sep='\t')
        return bool(df['eventType'].astype(str).str.startswith('sz').any())
    except Exception:
        return False

print('Iniciando download seletivo...')
s3 = get_s3_client()
VALID_SUBJECTS = []

for subject in tqdm(SUBJECTS, desc='Sujeitos'):
    keys = list_eeg_objects(s3, subject)
    tsv_keys = [k for k in keys if k.endswith('_events.tsv')]
    edfs_ok = 0
    for tsv_key in tsv_keys:
        local_tsv = s3_to_local(tsv_key)
        download_file(s3, tsv_key, local_tsv)
        if not tsv_has_seizure(local_tsv):
            continue
        edf_key = tsv_key.replace('_events.tsv', '_eeg.edf')
        if edf_key in keys:
            local_edf = s3_to_local(edf_key)
            download_file(s3, edf_key, local_edf)
            edfs_ok += 1
    if edfs_ok > 0:
        VALID_SUBJECTS.append(subject)
    tqdm.write(f'{subject}: {edfs_ok} EDF(s) com crise')

print(f'\n✅ Sujeitos válidos ({len(VALID_SUBJECTS)}): {VALID_SUBJECTS}')

Iniciando download seletivo...


Sujeitos:   8%|▊         | 1/12 [00:01<00:16,  1.51s/it]

sub-001: 4 EDF(s) com crise


Sujeitos:  17%|█▋        | 2/12 [00:01<00:07,  1.34it/s]

sub-002: 7 EDF(s) com crise


...

sub-010: 1 EDF(s) com crise


Sujeitos:  92%|█████████▏| 11/12 [00:04<00:00,  3.63it/s]

sub-011: 3 EDF(s) com crise


Sujeitos: 100%|██████████| 12/12 [00:04<00:00,  2.67it/s]

sub-012: 2 EDF(s) com crise

✅ Sujeitos válidos (12): ['sub-001', 'sub-002', 'sub-003', 'sub-004', 'sub-005', 'sub-006', 'sub-007', 'sub-008', 'sub-009', 'sub-010', 'sub-011', 'sub-012']


## 4. Carregamento e Rótulos

In [5]:
_NON_EEG = ['ecg','emg','acc','gyr','mov','resp','ekg','eog']

def load_eeg(edf_path):
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    try:
        raw.pick('eeg')
    except ValueError:
        keep = [ch for ch in raw.ch_names if not any(p in ch.lower() for p in _NON_EEG)]
        if keep: raw.pick(keep)
    return raw.get_data(), float(raw.info['sfreq']), raw.ch_names

def get_edf_paths(subject):
    eeg_dir = os.path.join(RAW_DIR, subject, SESSION, 'eeg')
    if not os.path.isdir(eeg_dir): return []
    return [
        os.path.join(eeg_dir, f)
        for f in sorted(os.listdir(eeg_dir))
        if f.endswith('_eeg.edf') and
           tsv_has_seizure(os.path.join(eeg_dir, f.replace('_eeg.edf','_events.tsv')))
    ]

def build_labels(n_samples, sfreq, tsv_path):
    df = pd.read_csv(tsv_path, sep='\t')
    labels = np.zeros(n_samples, dtype=np.int8)
    for _, row in df.iterrows():
        etype = str(row['eventType'])
        s = max(0, int(float(row['onset']) * sfreq))
        e = min(n_samples, int((float(row['onset']) + float(row['duration'])) * sfreq))
        if etype.startswith('sz'):   labels[s:e] = 1
        elif etype.startswith('impd'): labels[s:e] = 2
    return labels

print("✅ Funções de carregamento definidas.")

✅ Funções de carregamento definidas.


## 5. Pré-processamento

Cadeia: **passa-alta (0.5 Hz) → notch (50 Hz) → passa-baixa (40 Hz) → WOSG**

Sem Z-score — amplitudes reais são preservadas (RMS e Variância dependem delas).

In [6]:
def highpass(data, sfreq, cutoff=F_HIGH_PASS, order=4):
    sos = butter(order, cutoff, btype='high', fs=sfreq, output='sos')
    return sosfiltfilt(sos, data, axis=-1)

def notch(data, sfreq, freq=F_NOTCH, Q=30):
    b, a = iirnotch(freq, Q, sfreq)
    return filtfilt(b, a, data, axis=-1)

def lowpass(data, sfreq, cutoff=F_LOW_PASS, order=4):
    sos = butter(order, cutoff, btype='low', fs=sfreq, output='sos')
    return sosfiltfilt(sos, data, axis=-1)

def wosg(data, wavelet='db4', level=4, sg_win=11, sg_ord=3):
    n = data.shape[-1]
    out = np.zeros_like(data)
    for ch in range(data.shape[0]):
        coeffs = pywt.wavedec(data[ch], wavelet, level=level)
        new = [coeffs[0]]
        for d in coeffs[1:]:
            nd = len(d)
            w = min(sg_win, nd if nd % 2 == 1 else nd - 1)
            w = max(w, sg_ord + 2 if (sg_ord + 2) % 2 == 1 else sg_ord + 3)
            new.append(savgol_filter(d, w, sg_ord))
        out[ch] = pywt.waverec(new, wavelet)[:n]
    return out

def preprocess(data, sfreq):
    data = highpass(data, sfreq)
    data = notch(data, sfreq)
    data = lowpass(data, sfreq)
    data = wosg(data)
    return data

print("✅ Filtros definidos.")

✅ Filtros definidos.


## 6. Salvar Nível 1 — Sinal Pré-processado

In [8]:
print('Iniciando Nível 1...')
level1_index = {}

for subject in tqdm(VALID_SUBJECTS, desc='Sujeitos'):
    edfs = get_edf_paths(subject)
    saved = []
    for edf_path in edfs:
        run_id = next((p for p in os.path.basename(edf_path).split('_') if p.startswith('run')), 'run-XX')
        out = os.path.join(PREPROCESSED_DIR, subject, f'{subject}_{run_id}_preprocessed.npz')
        if os.path.exists(out):
            saved.append(out); continue
        data, sfreq, ch_names = load_eeg(edf_path)
        tsv = edf_path.replace('_eeg.edf', '_events.tsv')
        labels = build_labels(data.shape[1], sfreq, tsv)
        if (labels == 1).sum() == 0:
            tqdm.write(f'  AVISO {subject}/{run_id}: sem crise — pulando'); continue
        data = preprocess(data, sfreq)
        os.makedirs(os.path.dirname(out), exist_ok=True)
        np.savez(out, data=data.astype(np.float32), labels=labels,
                 sfreq=np.float32(sfreq), ch_names=np.array(ch_names))
        saved.append(out)
        tqdm.write(f'  ✓ {os.path.basename(out)}')
    level1_index[subject] = saved

print('\n✅ Nível 1 concluído.')
for s, ps in level1_index.items():
    print(f'  {s}: {len(ps)} gravação(ões)')

Iniciando Nível 1...


Sujeitos: 100%|██████████| 12/12 [00:00<00:00, 47.40it/s]


✅ Nível 1 concluído.
  sub-001: 4 gravação(ões)
  sub-002: 7 gravação(ões)
  sub-003: 1 gravação(ões)
  sub-004: 1 gravação(ões)
  sub-005: 2 gravação(ões)
  sub-006: 1 gravação(ões)
  sub-007: 1 gravação(ões)
  sub-008: 2 gravação(ões)
  sub-009: 1 gravação(ões)
  sub-010: 1 gravação(ões)
  sub-011: 3 gravação(ões)
  sub-012: 2 gravação(ões)


## 7. Segmentação — Regra das 3 Zonas

| Zona | Condição | Rótulo | Destino |
|------|----------|--------|---------|
| Seizure | ≥ 50% amostras com crise | 1 | ✅ Mantida |
| Non-Seizure | 0% amostras com crise, sem impd | 0 | ✅ Mantida |
| Transição / impd | demais | — | ❌ Descartada |

In [9]:
def segment(data, labels, sfreq, win_sec, overlap):
    ws   = int(win_sec * sfreq)
    step = int(ws * (1.0 - overlap))
    wins, wlbls = [], []
    for s in range(0, data.shape[1] - ws + 1, step):
        w = labels[s:s + ws]
        if np.any(w == 2): continue          # impd
        frac = (w == 1).mean()
        if frac >= SEIZURE_OVERLAP_THRESHOLD: lbl = 1
        elif frac == 0.0:                     lbl = 0
        else:                                 continue  # transição
        wins.append(data[:, s:s + ws])
        wlbls.append(lbl)
    if wins:
        return np.stack(wins).astype(np.float32), np.array(wlbls, dtype=np.int8)
    return np.empty((0, data.shape[0], ws), np.float32), np.empty(0, np.int8)

print("✅ Segmentação definida.")

✅ Segmentação definida.


## 8. ~~Undersampling~~ (removido)

**IMPORTANTE**: Undersampling será aplicado APENAS no Notebook 2 durante o treino.
Aqui no Nível 2, salvamos **TODAS as janelas** para permitir:
- Treino com ratio 1:3 ou 1:5
- Teste com paciente completo (validade clínica)

In [10]:
# Undersampling removido deste notebook.
# Será aplicado dinamicamente no Notebook 2 apenas no conjunto de treino.
print("✅ Undersampling será feito no Notebook 2 (apenas treino)")

✅ Undersampling será feito no Notebook 2 (apenas treino)


## 9. Salvar Nível 2 — Janelas Segmentadas

In [11]:
print('Iniciando Nível 2 (salvando TODAS as janelas)...')
level2_index = {cfg: {} for cfg in WINDOW_CONFIGS}

for subject in tqdm(VALID_SUBJECTS, desc='Sujeitos'):
    npz_paths = level1_index.get(subject, [])
    if not npz_paths: continue
    for cfg_name, cfg in WINDOW_CONFIGS.items():
        out_dir = os.path.join(WINDOWS_DIR, cfg_name, subject)
        dp = os.path.join(out_dir, f'{subject}_windows_data.npy')
        lp = os.path.join(out_dir, f'{subject}_windows_labels.npy')
        if os.path.exists(dp) and os.path.exists(lp):
            level2_index[cfg_name][subject] = (dp, lp); continue
        wins_list, lbls_list = [], []
        for npz_path in npz_paths:
            npz = np.load(npz_path)
            w, l = segment(npz['data'], npz['labels'], float(npz['sfreq']),
                           cfg['win_sec'], cfg['overlap'])
            wins_list.append(w); lbls_list.append(l)
        # Concatenar TODAS as janelas (sem undersampling)
        if not wins_list:
            continue
        all_w = np.concatenate(wins_list)
        all_l = np.concatenate(lbls_list)
        os.makedirs(out_dir, exist_ok=True)
        np.save(dp, all_w); np.save(lp, all_l)
        level2_index[cfg_name][subject] = (dp, lp)
        sz = int((all_l==1).sum()); non = int((all_l==0).sum())
        tqdm.write(f'  [{cfg_name}] {subject}: {sz} seizure | {non} non-sz (TODAS)')

print('\n✅ Nível 2 concluído (TODAS as janelas salvas).')

Iniciando Nível 2 (salvando TODAS as janelas)...


Sujeitos: 100%|██████████| 12/12 [00:00<00:00, 1648.43it/s]


✅ Nível 2 concluído (TODAS as janelas salvas).


## 10. Funções de Extração de Features

### GWST + Bandas EEG
**FS-A** (10/canal): L1-norm + Entropia de Shannon em 5 bandas via GWST  

In [12]:
EEG_BANDS = {
    'delta': (0.5,  4.0),
    'theta': (4.0,  8.0),
    'alpha': (8.0, 13.0),
    'beta' : (13.0, 30.0),
    'gamma': (30.0, 50.0),
}

# ── GWST ─────────────────────────────────────────────────────────────────
def precompute_gwst(n_samples, sfreq, f_min=0.5, f_max=50.0):
    df        = sfreq / n_samples
    fft_freqs = np.fft.fftfreq(n_samples, d=1.0 / sfreq)
    f_min_idx = max(1, int(np.ceil(f_min / df)))
    f_max_idx = int(np.floor(f_max / df))
    fidx      = np.arange(f_min_idx, f_max_idx + 1)
    freqs     = fidx * df
    G = np.exp(-2.0 * np.pi**2 * fft_freqs[np.newaxis, :]**2 / freqs[:, np.newaxis]**2)
    shift_idx = (np.arange(n_samples)[np.newaxis, :] + fidx[:, np.newaxis]) % n_samples
    return G, shift_idx, freqs

def gwst_abs(sig, G, shift_idx):
    H = np.fft.fft(sig.astype(np.float64))
    return np.abs(np.fft.ifft(H[shift_idx] * G, axis=1))

def band_mask(freqs, f_lo, f_hi):
    return (freqs >= f_lo) & (freqs < f_hi)

def gwst_l1(S, mask):
    return float(S[mask].sum()) if mask.any() else 0.0

def gwst_entropy(S, mask, bins=10):
    if not mask.any(): return 0.0
    v = S[mask].ravel()
    if v.max() == v.min(): return 0.0
    c, _ = np.histogram(v, bins=bins)
    p = c / c.sum(); p = p[p > 0]
    return float(-np.sum(p * np.log(p)))

# ── Features temporais ───────────────────────────────────────────────────
def temporal_features(sig):
    return np.array([
        float(np.var(sig)),
        float(scipy_kurtosis(sig, fisher=True, bias=False)),
        float(scipy_skew(sig, bias=False)),
        float(np.sqrt(np.mean(sig**2))),
        float(np.sum(np.diff(np.signbit(sig))) / len(sig)),
    ], np.float32)

def wavelet_energy(sig, wavelet='db4', level=5):
    coeffs = pywt.wavedec(sig.astype(np.float64), wavelet, level=level)
    feats = []
    for c in coeffs:
        feats.append(float(np.sum(c**2)))
        feats.append(float(np.var(c)))
    return np.array(feats, np.float32)

# ── Extração por window ───────────────────────────────────────────────────
def extract_fsa(window, G, shift_idx, freqs):
    """10 features/canal: 5 bandas × [L1-norm, Entropia]"""
    masks = {n: band_mask(freqs, lo, hi) for n, (lo, hi) in EEG_BANDS.items()}
    parts = []
    for ch in range(window.shape[0]):
        S = gwst_abs(window[ch], G, shift_idx)
        for mask in masks.values():
            parts.append(gwst_l1(S, mask))
            parts.append(gwst_entropy(S, mask))
    return np.array(parts, np.float32)

print(f'  FS-A: {5*2} features/canal')

  FS-A: 10 features/canal


### FS-B: Features Temporais e de Frequencia Adicionais

In [13]:
# ── Features adicionais para FS-B ────────────────────────────────────────
def temporal_features_extended(sig):
    """7 features temporais: media, variancia, skew, kurtosis, zero-cross, RMS, energia"""
    return np.array([
        float(np.mean(sig)),
        float(np.var(sig)),
        float(scipy_skew(sig, bias=False)),
        float(scipy_kurtosis(sig, fisher=True, bias=False)),
        float(np.sum(np.diff(np.signbit(sig))) / len(sig)),
        float(np.sqrt(np.mean(sig**2))),
        float(np.sum(sig**2)),
    ], np.float32)

def psd_band_features(sig, sfreq):
    """11 features de PSD: 5 potencias de banda + 5 razoes + freq dominante"""
    from scipy.signal import welch
    freqs, psd = welch(sig, fs=sfreq, nperseg=min(256, len(sig)))

    band_powers = {}
    for name, (lo, hi) in EEG_BANDS.items():
        idx = (freqs >= lo) & (freqs < hi)
        band_powers[name] = float(np.sum(psd[idx])) if idx.any() else 0.0

    ratios = [
        band_powers['alpha'] / (band_powers['delta'] + 1e-10),
        band_powers['theta'] / (band_powers['beta'] + 1e-10),
        band_powers['alpha'] / (band_powers['theta'] + 1e-10),
        band_powers['beta'] / (band_powers['alpha'] + 1e-10),
        band_powers['gamma'] / (band_powers['beta'] + 1e-10),
    ]
    dom_freq = float(freqs[np.argmax(psd)])
    return np.array(list(band_powers.values()) + ratios + [dom_freq], np.float32)

def entropy_features(sig):
    """2 features de entropia: Shannon + espectral"""
    hist, _ = np.histogram(sig, bins=20)
    hist = hist / (hist.sum() + 1e-10)
    hist = hist[hist > 0]
    shannon = float(-np.sum(hist * np.log(hist))) if len(hist) > 0 else 0.0

    psd = np.abs(np.fft.fft(sig))**2
    psd = psd / (psd.sum() + 1e-10)
    psd = psd[psd > 0]
    spectral = float(-np.sum(psd * np.log(psd))) if len(psd) > 0 else 0.0
    return np.array([shannon, spectral], np.float32)

def extract_fsb_additional(window, sfreq):
    """20 features adicionais/canal: 7 temp + 11 PSD + 2 entropia"""
    parts = []
    for ch in range(window.shape[0]):
        sig = window[ch]
        parts.extend(temporal_features_extended(sig))
        parts.extend(psd_band_features(sig, sfreq))
        parts.extend(entropy_features(sig))
    return np.array(parts, np.float32)

print('  FS-B adicional: 20 features/canal (7 temp + 11 PSD + 2 entropia)')

  FS-B adicional: 20 features/canal (7 temp + 11 PSD + 2 entropia)


### FS-C: Wavelet + Hjorth + Line Length

In [14]:
# ── Features para FS-C ────────────────────────────────────────────────────
def dwt_features(sig, wavelet='db4', level=5):
    """Features de DWT: energia + entropia + stats por nivel (D1-D5 + A5)"""
    coeffs = pywt.wavedec(sig.astype(np.float64), wavelet, level=level)
    feats = []
    for c in coeffs:
        energy = float(np.sum(c**2))
        feats.append(energy)
        hist, _ = np.histogram(c, bins=20)
        hist = hist / (hist.sum() + 1e-10)
        hist = hist[hist > 0]
        ent = float(-np.sum(hist * np.log(hist))) if len(hist) > 0 else 0.0
        feats.append(ent)
        feats.append(float(np.mean(c)))
        feats.append(float(np.var(c)))
        feats.append(float(np.std(c)))
    return np.array(feats, np.float32)

def hjorth_parameters(sig):
    """3 features: Activity, Mobility, Complexity"""
    activity = float(np.var(sig))
    d1 = np.diff(sig)
    var_d1 = np.var(d1)
    mobility = float(np.sqrt(var_d1 / (activity + 1e-10)))
    d2 = np.diff(d1)
    var_d2 = np.var(d2)
    complexity = float(np.sqrt(var_d2 / (var_d1 + 1e-10)) / (mobility + 1e-10))
    return np.array([activity, mobility, complexity], np.float32)

def line_length(sig):
    """1 feature: soma das diferencas absolutas"""
    return float(np.sum(np.abs(np.diff(sig))))

def extract_fsc(window, G=None, shift_idx=None, freqs=None):
    """Features wavelet/canal: 30 DWT + 3 Hjorth + 1 Line Length = 34/canal"""
    parts = []
    for ch in range(window.shape[0]):
        sig = window[ch]
        parts.extend(dwt_features(sig))
        parts.extend(hjorth_parameters(sig))
        parts.append(line_length(sig))
    return np.array(parts, np.float32)

print('  FS-C: 34 features/canal (30 DWT + 3 Hjorth + 1 Line Length)')

  FS-C: 34 features/canal (30 DWT + 3 Hjorth + 1 Line Length)


## 11. Extração de Features — FS-A (sequencial)

Extrai um feature set de cada vez, para todas as janelas e todos os sujeitos.

In [15]:
def extract_and_save(fs_name, extractor, cfg_names=None, subjects=None, batch=64):
    """
    Extrai features e salva em ./data/features/<fs_name>/<cfg>/<subject>_features.npy
    Retorna level3_index[cfg_name][subject] = (feat_path, labels_path)
    """
    if cfg_names is None: cfg_names = list(WINDOW_CONFIGS.keys())
    if subjects is None:  subjects  = list(VALID_SUBJECTS)
    index = {cfg: {} for cfg in cfg_names}

    for cfg_name in cfg_names:
        cfg      = WINDOW_CONFIGS[cfg_name]
        ws       = int(cfg['win_sec'] * SFREQ)
        feat_dir = os.path.join(FEATURES_DIR, fs_name, cfg_name)
        os.makedirs(feat_dir, exist_ok=True)
        G, shift_idx, freqs = precompute_gwst(ws, SFREQ)
        print(f'\n[{fs_name}][{cfg_name}] win={ws} amostras')

        for subject in tqdm(subjects, desc=f'{fs_name}/{cfg_name}'):
            entry = level2_index[cfg_name].get(subject)
            if entry is None: continue
            dp, lp = entry
            out_f = os.path.join(feat_dir, f'{subject}_features.npy')
            out_l = os.path.join(feat_dir, f'{subject}_labels.npy')
            if os.path.exists(out_f) and os.path.exists(out_l):
                index[cfg_name][subject] = (out_f, out_l)
                tqdm.write(f'  {subject}: já existe — pulando'); continue

            windows = np.load(dp, mmap_mode='r')
            labels  = np.load(lp)
            n_win   = len(labels)
            # dimensão de uma feature
            n_feat  = len(extractor(np.array(windows[0]), G, shift_idx, freqs))
            out_tmp = out_f.replace('.npy', '_tmp.npy')
            mm = np.lib.format.open_memmap(out_tmp, 'w+', np.float32, (n_win, n_feat))
            tqdm.write(f'  {subject}: {n_win} janelas × {n_feat} features [{fs_name}]')
            for start in tqdm(range(0, n_win, batch), desc=f'  {subject}', leave=False):
                end = min(start + batch, n_win)
                b   = np.array(windows[start:end])
                for i, w in enumerate(b):
                    mm[start + i] = extractor(w, G, shift_idx, freqs)
                mm.flush(); del b
            del mm, windows; gc.collect()
            os.replace(out_tmp, out_f)
            np.save(out_l, labels)
            index[cfg_name][subject] = (out_f, out_l)

    return index

print("✅ Função de extração genérica definida.")

✅ Função de extração genérica definida.


### 11.1 Extrair FS-A

In [16]:
print('='*60)
print('Extraindo FS-A...')
print('='*60)
level3_fsa = extract_and_save('FS-A', extract_fsa)
print('\n✅ FS-A concluído.')

Extraindo FS-A...

[FS-A][4s_50] win=1000 amostras


FS-A/4s_50: 100%|██████████| 12/12 [00:00<00:00, 81.26it/s]

  sub-001: já existe — pulando
  sub-002: já existe — pulando
  sub-003: já existe — pulando
  sub-004: já existe — pulando
  sub-005: já existe — pulando
  sub-006: já existe — pulando
  sub-007: já existe — pulando
  sub-008: já existe — pulando
  sub-009: já existe — pulando
  sub-010: já existe — pulando
  sub-011: já existe — pulando
  sub-012: já existe — pulando



[FS-A][5s_50] win=1250 amostras


FS-A/5s_50:   0%|          | 0/12 [00:00<?, ?it/s]

...

  sub-001: já existe — pulando
  sub-002: já existe — pulando
  sub-003: já existe — pulando
  sub-004: já existe — pulando
  sub-005: já existe — pulando
  sub-006: já existe — pulando
  sub-007: já existe — pulando
  sub-008: já existe — pulando


FS-A/6s_50:   0%|          | 0/12 [00:00<?, ?it/s]

  sub-009: já existe — pulando
  sub-010: já existe — pulando
  sub-011: já existe — pulando
  sub-012: já existe — pulando


FS-A/6s_50: 100%|██████████| 12/12 [00:00<00:00, 249.26it/s]


✅ FS-A concluído.


### 11.2 Extrair FS-B (FS-A + features adicionais)

**Estrategia**: Carregar FS-A salvo e concatenar com features adicionais (sem recalcular FS-A).

In [17]:
print('='*60)
print('Extraindo FS-B (FS-A + adicionais)...')
print('='*60)

level3_fsb = {cfg: {} for cfg in WINDOW_CONFIGS}
batch = 64

for cfg_name in WINDOW_CONFIGS:
    feat_dir = os.path.join(FEATURES_DIR, 'FS-B', cfg_name)
    os.makedirs(feat_dir, exist_ok=True)

    for subject in tqdm(VALID_SUBJECTS, desc=f'FS-B/{cfg_name}'):
        entry = level2_index[cfg_name].get(subject)
        if entry is None:
            continue
        dp, lp = entry

        out_f = os.path.join(feat_dir, f'{subject}_features.npy')
        out_l = os.path.join(feat_dir, f'{subject}_labels.npy')
        if os.path.exists(out_f) and os.path.exists(out_l):
            level3_fsb[cfg_name][subject] = (out_f, out_l)
            tqdm.write(f'  {subject}: FS-B ja existe — pulando')
            continue

        fsa_dir = os.path.join(FEATURES_DIR, 'FS-A', cfg_name)
        fsa_f = os.path.join(fsa_dir, f'{subject}_features.npy')
        fsa_l = os.path.join(fsa_dir, f'{subject}_labels.npy')
        if not (os.path.exists(fsa_f) and os.path.exists(fsa_l)):
            tqdm.write(f'  {subject}: FS-A nao encontrado — pulando')
            continue

        X_fsa = np.load(fsa_f, mmap_mode='r')
        labels = np.load(fsa_l)
        windows = np.load(dp, mmap_mode='r')
        n_win = len(labels)
        if len(windows) != n_win:
            tqdm.write(f'  {subject}: mismatch janelas/labels — pulando')
            del X_fsa, labels, windows; gc.collect()
            continue

        n_add = len(extract_fsb_additional(np.array(windows[0]), SFREQ))
        n_total = X_fsa.shape[1] + n_add
        out_tmp = out_f.replace('.npy', '_tmp.npy')
        mm = np.lib.format.open_memmap(out_tmp, 'w+', np.float32, (n_win, n_total))
        tqdm.write(f'  {subject}: {n_win} janelas × {n_total} features [FS-B]')
        for start in tqdm(range(0, n_win, batch), desc=f'  {subject}', leave=False):
            end = min(start + batch, n_win)
            b = np.array(windows[start:end])
            X_add = np.array([extract_fsb_additional(w, SFREQ) for w in b], np.float32)
            mm[start:end, :X_fsa.shape[1]] = X_fsa[start:end]
            mm[start:end, X_fsa.shape[1]:] = X_add
            mm.flush()
            del b, X_add
        del mm, windows, X_fsa; gc.collect()
        os.replace(out_tmp, out_f)
        np.save(out_l, labels)
        level3_fsb[cfg_name][subject] = (out_f, out_l)

print('\n✅ FS-B concluido.')

Extraindo FS-B (FS-A + adicionais)...


FS-B/4s_50:   0%|          | 0/12 [00:00<?, ?it/s]

  sub-001: 104049 janelas × 60 features [FS-B]


FS-B/4s_50:   8%|▊         | 1/12 [08:04<1:28:44, 484.08s/it]

  sub-002: 134089 janelas × 60 features [FS-B]


...

  sub-011: 21577 janelas × 60 features [FS-B]


FS-B/6s_50:  92%|█████████▏| 11/12 [18:22<00:53, 53.09s/it]

  sub-012: 32430 janelas × 60 features [FS-B]


FS-B/6s_50: 100%|██████████| 12/12 [20:23<00:00, 101.96s/it]


✅ FS-B concluido.


### 11.3 Extrair FS-C

In [18]:
print('='*60)
print('Extraindo FS-C...')
print('='*60)
level3_fsc = extract_and_save('FS-C', extract_fsc)
print('\n✅ FS-C concluido.')

Extraindo FS-C...

[FS-C][4s_50] win=1000 amostras


FS-C/4s_50:   0%|          | 0/12 [00:00<?, ?it/s]

  sub-001: 104049 janelas × 68 features [FS-C]


FS-C/4s_50:   8%|▊         | 1/12 [04:24<48:34, 264.94s/it]

  sub-002: 134089 janelas × 68 features [FS-C]


...

  sub-011: 21577 janelas × 68 features [FS-C]


FS-C/6s_50:  92%|█████████▏| 11/12 [17:22<00:44, 44.76s/it]

  sub-012: 32430 janelas × 68 features [FS-C]


FS-C/6s_50: 100%|██████████| 12/12 [18:54<00:00, 94.51s/it]


✅ FS-C concluido.


## 12. Salvar Índices para o Notebook 2

In [21]:
import json

def index_to_json(idx):
    return {cfg: {sub: list(paths) for sub, paths in subs.items()} for cfg, subs in idx.items()}

meta = {
    'VALID_SUBJECTS': VALID_SUBJECTS,
    'WINDOW_CONFIGS': {k: v for k, v in WINDOW_CONFIGS.items()},
    'SFREQ': SFREQ,
    'DATA_DIR': DATA_DIR,
    'level3_fsa': index_to_json(level3_fsa),
    'level3_fsb': index_to_json(level3_fsb),
    'level3_fsc': index_to_json(level3_fsc),
}

meta_path = os.path.join(DATA_DIR, 'pipeline_meta.json')
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)

print(f'✅ Índices salvos em {meta_path}')
print(f'   Sujeitos válidos: {VALID_SUBJECTS}')

def print_summary(tag, idx):
    for cfg, subs in idx.items():
        total = sum(len(np.load(lp)) for _, lp in subs.values())
        print(f'  [{tag}][{cfg}] {len(subs)} sujeitos | {total:,} janelas')

# Resumo de janelas por configuração
print_summary('FS-A', level3_fsa)
print_summary('FS-B', level3_fsb)
print_summary('FS-C', level3_fsc)

✅ Índices salvos em ./data\pipeline_meta.json
   Sujeitos válidos: ['sub-001', 'sub-002', 'sub-003', 'sub-004', 'sub-005', 'sub-006', 'sub-007', 'sub-008', 'sub-009', 'sub-010', 'sub-011', 'sub-012']
  [FS-A][4s_50] 12 sujeitos | 481,620 janelas
  [FS-A][5s_50] 12 sujeitos | 385,238 janelas
  [FS-A][6s_50] 12 sujeitos | 321,026 janelas
  [FS-B][4s_50] 12 sujeitos | 481,620 janelas
  [FS-B][5s_50] 12 sujeitos | 385,238 janelas
  [FS-B][6s_50] 12 sujeitos | 321,026 janelas
  [FS-C][4s_50] 12 sujeitos | 481,620 janelas
  [FS-C][5s_50] 12 sujeitos | 385,238 janelas
  [FS-C][6s_50] 12 sujeitos | 321,026 janelas


## 13. Atualizar Indices com FS-B/FS-C

In [22]:
meta_path = os.path.join(DATA_DIR, 'pipeline_meta.json')
if os.path.exists(meta_path):
    with open(meta_path, 'r') as f:
        meta = json.load(f)
    meta['level3_fsa'] = index_to_json(level3_fsa)
    meta['level3_fsb'] = index_to_json(level3_fsb)
    meta['level3_fsc'] = index_to_json(level3_fsc)
    with open(meta_path, 'w') as f:
        json.dump(meta, f, indent=2)
    print(f'✅ Indices FS-A/FS-B/FS-C atualizados em {meta_path}')
else:
    print('AVISO: pipeline_meta.json nao encontrado. Rode a secao 12 antes.')

✅ Indices FS-A/FS-B/FS-C atualizados em ./data\pipeline_meta.json
